In [4]:
@dataclass
class Hardware:
    name: str
    flops: int
    bw: int
    w_nvlink: int
    w_rdma: int
    p: int

@dataclass
class Model:
    name: str
    m_compute: int
    m_params: int
    hidden_dim: int
    q_head: int
    kv_head: int
    head_dim: int
    layer_num: int

@dataclass
class Dtype:
    compute_factor: int
    w_factor: int
    

hardware_map = {
    "H100": Hardware("H100", 989, 3350, 450, 50, 8),
    "A100": Hardware("A100", 312, 2000, 200, 25, 8),
    "H20": Hardware("H20", 148, 4000, 450, 50, 8),
    "L40s": Hardware("L40s", 362, 864, 25, 25, 1),
    "L20": Hardware("L20", 120, 864, 25, 25, 1),
}

model_map = {
    "qwen3-32B": Model("qwen3-32B", 32, 32, 5120, 64, 8, 128, 64),
    "qwen3-30B-A3B": Model("qwen3-30B-A3B", 3, 30, 2048, 32, 4, 128, 48),
    "qwen3-235B-A22B": Model("qwen3-235B-A22B", 22, 235, 4096, 64, 4, 128, 94),
    "deepseek-v3": Model("deepseek-v3", 30, 671, 7168, 128, 4, 128, 61),
}

dtype_map = {
    "f32": Dtype(0.5, 4),
    "bf16": Dtype(1, 2),
    "f8": Dtype(2, 1),
}

In [6]:
def batch_dense_model(hardware, dtype):
    h = hardware_map[hardware]
    d = dtype_map[dtype]

    b = d.w_factor * h.flops * 1e12 / (3 * h.w_nvlink * 1e9)

    print(f"|{hardware}|{dtype}|{b:.0f}|")

batch_dense_model("H100", "f32")
batch_dense_model("H100", "bf16")
batch_dense_model("H100", "f8")
batch_dense_model("A100", "f32")
batch_dense_model("A100", "bf16")
batch_dense_model("H20", "f32")
batch_dense_model("H20", "bf16")
batch_dense_model("H20", "f8")

|H100|f32|2930|
|H100|bf16|1465|
|H100|f8|733|
|A100|f32|2080|
|A100|bf16|1040|
|H20|f32|439|
|H20|bf16|219|
|H20|f8|110|


In [28]:
def batch_moe_model(hardware, dtype, model):
    h = hardware_map[hardware]
    d = dtype_map[dtype]
    m = model_map[model]

    b = d.w_factor * h.flops * d.compute_factor * 1e12 * m.m_params / (3 * h.w_nvlink * 1e9 * m.m_compute)

    print(f"|{hardware}|{dtype}|{model}|{b:.0f}|")


batch_moe_model("H100", "f32", "qwen3-32B")
batch_moe_model("H100", "f32", "qwen3-30B-A3B")
batch_moe_model("H100", "f32", "qwen3-235B-A22B")
batch_moe_model("H100", "f8", "deepseek-v3")

|H100|f32|qwen3-32B|1465|
|H100|f32|qwen3-30B-A3B|14652|
|H100|f32|qwen3-235B-A22B|15651|
|H100|f8|deepseek-v3|32771|


In [35]:
def batch_moe_memory(hardware, dtype, model, chips):
    h = hardware_map[hardware]
    d = dtype_map[dtype]
    m = model_map[model]

    remaining = (80-15)*1e9 - (2 * d.w_factor + 8) * m.m_params * 1e9 / chips - (2 * m.m_params * d.w_factor * 1e9)/m.layer_num
    b = remaining / (m.hidden_dim * m.layer_num * d.w_factor)

    print(f"|{hardware}|{chips}|{dtype}|{model}|{b:.0f}|")


batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 128)
batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 128)
batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 256)
batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 512)
batch_moe_memory("H100", "bf16", "qwen3-235B-A22B", 512)
batch_moe_memory("H100", "f8", "deepseek-v3", 256)
# batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 128)

|H100|128|f32|qwen3-235B-A22B|10145|
|H100|128|f32|qwen3-235B-A22B|10145|
|H100|256|f32|qwen3-235B-A22B|19682|
|H100|512|f32|qwen3-235B-A22B|24451|
|H100|512|bf16|qwen3-235B-A22B|64272|
|H100|256|f8|deepseek-v3|38397|


In [ ]:
def tp_accm(hardware, dtype, model, b):
    h = hardware_map[hardware]
    d = dtype_map[dtype]
    m = model_map[model]

    comm_w = 2 * m.m_params * d.w_factor * 1e9 / (h.w_nvlink*1e9)

    acc_f = 2 * b * m.

    remaining = (80-15)*1e9 - (2 * d.w_factor + 8) * m.m_params * 1e9 / chips - (2 * m.m_params * d.w_factor * 1e9)/m.layer_num
    b = remaining / (m.hidden_dim * m.layer_num * d.w_factor)

    print(f"|{hardware}|{chips}|{dtype}|{model}|{b:.0f}|")


batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 128)
batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 128)
batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 256)
batch_moe_memory("H100", "f32", "qwen3-235B-A22B", 512)
batch_moe_memory("H100", "bf16", "qwen3-235B-A22B", 512)